# Wk1: Properties filtered to residental

In [ ]:
from ast import List
import csv
from pathlib import Path
from re import match
import struct
from types import SimpleNamespace
import numpy as np
import pandas as pd

ClosePrice= SimpleNamespace(Column=-1,Min=0, Max=0,Mean=0,median=0, percentiles=0)
LivingArea= SimpleNamespace(Column=-1,Min=0, Max=0,Mean=0,median=0, percentiles=0)
DaysOnMarket= SimpleNamespace(Column=-1,Max=0,Mean=0,median=0, percentiles=0)
Column = SimpleNamespace(proptype=-1, closeprice=-1, livingarea=-1, daysonmarket=-1)


# Inspect structure
#prefix.columns
#sold.head()
# Check property categories
#sold['PropertyType'].unique()
# Filter residential
#sold = sold[sold.PropertyType == 'Residential']
# Validate completeness


def Write_Calcs(file_path, file_name,Column,rows):
    for prefix in (ClosePrice, LivingArea, DaysOnMarket):
        #maybe do calcs functionand then return them for writing

        prefix= AllCalcs(prefix.Min, prefix.Max, prefix.Mean, prefix.median, prefix.percentiles)
        NullColumns=prefix.isnull().sum() #It returns the total number of missing or null values (such as NaN or None) in a specific column or dataset named sold
        UniqueTypes=prefix['PropertyType'].unique()
        List.clear() #resets array list after done


def AllCalcs(Catagory,List):
    return np.min(List), np.max(List), np.mean(List), np.median(List), np.percentile(List, 90)

def Typefiltering(row, Column, filtered_rows):
    # Take a single CSV row, and append to filtered_rows if PropertyType == 'Residential'.
    try:
        if row and Column.proptype >= 0 and len(row) > Column.proptype and row[Column.proptype].strip() == "Residential":
            filtered_rows.append(row)
    except Exception:
        pass

    return filtered_rows

def read_csv_file(file_path, file_name,Column):
    file_path = Path(file_path)
    filtered_rows = []
    List= [] #all elements of array are zero
    try:
        if not file_path.exists():
            print(f"CSV folder does not exist: {file_path}")
            return filtered_rows
        pattern = f"{file_name}*.csv"
        for csv_file in file_path.glob(pattern):  # goes through all files in the directory that match the pattern
            name_l = csv_file.name.lower()
            if name_l.endswith("_filtered.csv") or name_l.endswith("_calculated.csv"):
                continue # exclude previously generated files

            with csv_file.open("r", newline="", encoding="utf-8") as file:
                reader = csv.reader(file)
                for row in reader:
                    filtered_rows = Typefiltering(row, Column, filtered_rows)

            # after reading this file, print any filtered rows found
            if filtered_rows:
                print(f"Found {len(filtered_rows)} residential rows in {csv_file.name}")
               # print_filtered_csv(file_path, file_name, Column, filtered_rows)

        # return collected filtered rows for the caller
        return filtered_rows

        
    except FileNotFoundError:
        print("File not found. Please check the file path and try again.")
        print(file_path)
        return filtered_rows
    except PermissionError:
        print("Please check the file permissions and try again.")
        return filtered_rows


def print_filtered_csv(file_path, file_name,Column,rows):
   
    # rows = read_csv_file(file_path, file_name,Column)
     for row in rows:
        print(",".join(row)) #prints all the rows in the filtered list as a single string, with each value separated by a comma
    # return rows


def create_csv_file(file_path, file_name,Column):
    rows = read_csv_file(file_path, file_name,Column)
    file_path = Path(file_path)
    if not file_path.exists():
        try:
            file_path.mkdir(parents=True, exist_ok=True)
            print(f"Created output directory: {file_path}")
        except Exception as e:
            print(f"Could not create output directory: {e}")
            return
    for prefix in ("_calculated.csv", "_filtered.csv"):
        output_path = Path(file_path) / f"{file_name}{prefix}"
 
        try:
            with output_path.open("w", newline="", encoding="utf-8") as file:
                writer = csv.writer(file)
                writer.writerows(rows)
                #make sure all records are on their own rows
            print(f"\nFiltered rows written to {output_path}")
        except PermissionError:
            print("Please check the write permissions and try again.")
        except Exception as e:
            print(f"An error occurred: {e}")


if __name__ == "__main__":
    file_path = Path("C:/Users/Viv/Documents/Career readiness/internships/IDX exchange/csv")

    for prefix in ("CRMLSListing", "CRMLSSold"):
        column = -1 #allows for Property type column be located in any column

        match prefix:
            case "CRMLSListing":
                Column.proptype = 10  # property type column in the CSV file
                Column.closeprice = 4
                Column.livingarea = 11
                Column.daysonmarket = 13

            case "CRMLSSold":
                Column.proptype = 17
                Column.closeprice = 11  # one minus actual column number
                Column.livingarea = 18
                Column.daysonmarket = 20

            case _:
                print(f"Column for {prefix} is not vaild. Please check the column index and try again.")
                continue

        if (Column.proptype > -1 and Column.closeprice > -1 and Column.livingarea > -1 and Column.daysonmarket > -1):
            create_csv_file(file_path, prefix, Column)
           # create_csv_file(file_path, prefix, column)

Found 17007 residential rows in CRMLSListing202401.csv
Found 34497 residential rows in CRMLSListing202402.csv
Found 54998 residential rows in CRMLSListing202403.csv
Found 79023 residential rows in CRMLSListing202404.csv
Found 104470 residential rows in CRMLSListing202405.csv
Found 127780 residential rows in CRMLSListing202406.csv
Found 150799 residential rows in CRMLSListing202407.csv
Found 173014 residential rows in CRMLSListing202408.csv
Found 195271 residential rows in CRMLSListing202409.csv
Found 217192 residential rows in CRMLSListing202410.csv
Found 232300 residential rows in CRMLSListing202411.csv
Found 242994 residential rows in CRMLSListing202412.csv
Found 265684 residential rows in CRMLSListing202501.csv
Found 287614 residential rows in CRMLSListing202502.csv
Found 312718 residential rows in CRMLSListing202503.csv
Found 339413 residential rows in CRMLSListing202504.csv
Found 365851 residential rows in CRMLSListing202505.csv
Found 382907 residential rows in CRMLSListing202506.

# WK2-3: Data vaildation

In [4]:
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd


SOURCE_DIR = Path("C:/Users/Viv/Documents/Career readiness/internships/IDX exchange/csv")
START_YEAR = 2024
START_MONTH = 1
PROPERTY_FILTER = "Residential"
PERCENTILES = (10, 25, 50, 75, 90)
TERMINAL_FLOAT_FORMAT = lambda value: f"{value:,.6f}"


# Reporting rules:
# 1. Build a combined dataset from all monthly files in the date range.
# 2. Report unique PropertyType values found before filtering.
# 3. Apply the Residential-only filter.
# 4. Print and save null-count, missing-value, and numeric distribution summaries.
# 5. Save the filtered dataset as a new CSV.


def most_recent_completed_month(today=None):
    today = today or date.today()
    first_day_of_current_month = date(today.year, today.month, 1)
    last_completed_month = first_day_of_current_month - pd.offsets.MonthBegin(1)
    return last_completed_month.year, last_completed_month.month


def month_range(start_year, start_month, end_year, end_month):
    start = pd.Period(f"{start_year:04d}-{start_month:02d}", freq="M")
    end = pd.Period(f"{end_year:04d}-{end_month:02d}", freq="M")
    for period in pd.period_range(start, end, freq="M"):
        yield period.year, period.month


def build_monthly_file_list(prefix, start_year, start_month, end_year, end_month):
    monthly_files = []
    for year, month in month_range(start_year, start_month, end_year, end_month):
        file_name = f"{prefix}{year:04d}{month:02d}.csv"
        file_path = SOURCE_DIR / file_name
        if file_path.exists():
            monthly_files.append(file_path)
    return monthly_files


def load_monthly_data(files):
    frames = []
    row_count_before_concat = 0

    for file_path in files:
        frame = pd.read_csv(file_path, low_memory=False)
        frames.append(frame)
        row_count_before_concat += len(frame)

    print(f"Row count before concatenation: {row_count_before_concat}")

    if frames:
        combined = pd.concat(frames, ignore_index=True)
    else:
        combined = pd.DataFrame()

    print(f"Row count after concatenation: {len(combined)}")
    return combined


def get_property_column(frame):
    if "PropertyType" not in frame.columns:
        raise KeyError("PropertyType column was not found in the combined dataset.")
    return "PropertyType"


def unique_property_types(frame, property_column):
    values = frame[property_column].dropna().astype(str).str.strip()
    values = values[values != ""]
    return sorted(values.unique().tolist())


def filter_residential_only(frame, property_column):
    print(f"Filtering logic applied: keep rows where {property_column} equals {PROPERTY_FILTER}.")
    before_filter = len(frame)
    filtered = frame[frame[property_column].astype(str).str.strip() == PROPERTY_FILTER].copy()
    print(f"Row count before Residential filter: {before_filter}")
    print(f"Row count after Residential filter: {len(filtered)}")
    return filtered


def build_null_summary(frame):
    summary = pd.DataFrame({
        "column": frame.columns,
        "null_count": frame.isna().sum().values,
        "row_count": len(frame),
    })
    summary["null_percent"] = np.where(
        summary["row_count"] > 0,
        (summary["null_count"] / summary["row_count"]) * 100,
        0.0,
    )
    return summary[["column", "null_count", "null_percent"]]


def build_missing_value_report(null_summary):
    flagged = null_summary[null_summary["null_percent"] > 90].copy()
    flagged.insert(0, "flag", "above_90_percent_null")
    return flagged[["flag", "column", "null_count", "null_percent"]]


def build_numeric_distribution_summary(frame, fields):
    rows = []
    for field in fields:
        if field not in frame.columns:
            rows.extend([
                {"field": field, "metric": "min", "value": np.nan},
                {"field": field, "metric": "max", "value": np.nan},
                {"field": field, "metric": "mean", "value": np.nan},
                {"field": field, "metric": "median", "value": np.nan},
                {"field": field, "metric": "p10", "value": np.nan},
                {"field": field, "metric": "p25", "value": np.nan},
                {"field": field, "metric": "p50", "value": np.nan},
                {"field": field, "metric": "p75", "value": np.nan},
                {"field": field, "metric": "p90", "value": np.nan},
            ])
            continue

        values = pd.to_numeric(frame[field], errors="coerce").dropna()
        if values.empty:
            stats = {
                "min": np.nan,
                "max": np.nan,
                "mean": np.nan,
                "median": np.nan,
                "p10": np.nan,
                "p25": np.nan,
                "p50": np.nan,
                "p75": np.nan,
                "p90": np.nan,
            }
        else:
            percentiles = np.percentile(values, PERCENTILES)
            stats = {
                "min": values.min(),
                "max": values.max(),
                "mean": values.mean(),
                "median": values.median(),
                "p10": percentiles[0],
                "p25": percentiles[1],
                "p50": percentiles[2],
                "p75": percentiles[3],
                "p90": percentiles[4],
            }

        for metric, value in stats.items():
            rows.append({"field": field, "metric": metric, "value": value})

    return pd.DataFrame(rows)


def build_report(file_name, frame, filtered_frame):
    property_column = get_property_column(frame)
    unique_types = unique_property_types(frame, property_column)
    null_summary = build_null_summary(frame)
    missing_value_report = build_missing_value_report(null_summary)
    numeric_summary = build_numeric_distribution_summary(
        frame,
        ["ClosePrice", "LivingArea", "DaysOnMarket"],
    )

    print(f"Unique property types found for {file_name}: {unique_types}")

    report_rows = []
    report_rows.append({
        "section": "metadata",
        "item": "file_name",
        "metric": "value",
        "value": file_name,
    })
    report_rows.append({
        "section": "metadata",
        "item": "source_rows",
        "metric": "count",
        "value": len(frame),
    })
    report_rows.append({
        "section": "metadata",
        "item": "filtered_rows",
        "metric": "count",
        "value": len(filtered_frame),
    })
    report_rows.append({
        "section": "filtering_logic",
        "item": property_column,
        "metric": "rule",
        "value": f'keep rows where value equals "{PROPERTY_FILTER}"',
    })
    report_rows.append({
        "section": "unique_property_types",
        "item": property_column,
        "metric": "unique_values",
        "value": ", ".join(unique_types),
    })

    for _, row in null_summary.iterrows():
        report_rows.append({
            "section": "null_summary",
            "item": row["column"],
            "metric": "null_count",
            "value": row["null_count"],
        })
        report_rows.append({
            "section": "null_summary",
            "item": row["column"],
            "metric": "null_percent",
            "value": round(float(row["null_percent"]), 2),
        })

    for _, row in missing_value_report.iterrows():
        report_rows.append({
            "section": "missing_value_report",
            "item": row["column"],
            "metric": row["flag"],
            "value": round(float(row["null_percent"]), 2),
        })

    for _, row in numeric_summary.iterrows():
        report_rows.append({
            "section": "numeric_distribution",
            "item": row["field"],
            "metric": row["metric"],
            "value": row["value"],
        })

    report_frame = pd.DataFrame(report_rows)
    print("\nNull-count summary table:")
    print(null_summary.to_string(index=False, float_format=TERMINAL_FLOAT_FORMAT))
    print("\nMissing value report (columns above 90% null):")
    print(missing_value_report.to_string(index=False, float_format=TERMINAL_FLOAT_FORMAT))
    print("\nNumeric distribution summary:")
    print(numeric_summary.to_string(index=False, float_format=TERMINAL_FLOAT_FORMAT))
    return report_frame


def save_outputs(file_path, file_name, filtered_frame, report_frame):
    file_path = Path(file_path)
    file_path.mkdir(parents=True, exist_ok=True)

    filtered_output = file_path / f"{file_name}_filtered_residential.csv"
    report_output = file_path / f"{file_name}_calculated_summary.csv"

    filtered_frame.to_csv(filtered_output, index=False)
    report_frame.to_csv(report_output, index=False)

    print(f"\nFiltered dataset saved to {filtered_output}")
    print(f"Summary report saved to {report_output}")


def process_prefix(prefix):
    end_year, end_month = most_recent_completed_month()
    monthly_files = build_monthly_file_list(prefix, START_YEAR, START_MONTH, end_year, end_month)

    if not monthly_files:
        print(f"No monthly files found for {prefix}.")
        return

    print(
        f"Found {len(monthly_files)} monthly files for {prefix} "
        f"from {START_YEAR:04d}-{START_MONTH:02d} through {end_year:04d}-{end_month:02d}."
    )

    combined = load_monthly_data(monthly_files)
    if combined.empty:
        print(f"Combined dataset for {prefix} is empty.")
        return

    filtered = filter_residential_only(combined, get_property_column(combined))
    report_frame = build_report(prefix, combined, filtered)
    save_outputs(SOURCE_DIR, prefix, filtered, report_frame)


if __name__ == "__main__":
    for prefix in ("CRMLSListing", "CRMLSSold"):
        process_prefix(prefix)

Found 28 monthly files for CRMLSListing from 2024-01 through 2026-06.
Row count before concatenation: 893594
Row count after concatenation: 893594
Filtering logic applied: keep rows where PropertyType equals Residential.
Row count before Residential filter: 893594
Row count after Residential filter: 567549
Unique property types found for CRMLSListing: ['BusinessOpportunity', 'CommercialLease', 'CommercialSale', 'Land', 'ManufacturedInPark', 'Residential', 'ResidentialIncome', 'ResidentialLease']

Null-count summary table:
                      column  null_count  null_percent
           OriginalListPrice        3444      0.385410
                  ListingKey           0      0.000000
              ListAgentEmail        2372      0.265445
                   CloseDate      635609     71.129506
                  ClosePrice      660222     73.883889
          ListAgentFirstName        5280      0.590872
           ListAgentLastName          83      0.009288
                    Latitude    

# WK2-3: Data from FRED merged onto sold and listings datasets using a year_month key

In [ ]:
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd


SOURCE_DIR = Path("C:/Users/Viv/Documents/Career readiness/internships/IDX exchange/csv")
START_YEAR = 2024
START_MONTH = 1
PROPERTY_FILTER = "Residential"
PERCENTILES = (10, 25, 50, 75, 90)
TERMINAL_FLOAT_FORMAT = lambda value: f"{value:,.6f}"

# Step 1 – Fetch the mortgage rate data from FRED
URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(URL, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']


# Reporting rules:
# 1. Build a combined dataset from all monthly files in the date range.
# 2. Report unique PropertyType values found before filtering.
# 3. Apply the Residential-only filter.
# 4. Print and save null-count, missing-value, and numeric distribution summaries.
# 5. Save the filtered dataset as a new CSV.
#hi

def most_recent_completed_month(today=None):
    today = today or date.today()
    first_day_of_current_month = date(today.year, today.month, 1)
    last_completed_month = first_day_of_current_month - pd.offsets.MonthBegin(1)
    return last_completed_month.year, last_completed_month.month


def month_range(start_year, start_month, end_year, end_month):
    start = pd.Period(f"{start_year:04d}-{start_month:02d}", freq="M")
    end = pd.Period(f"{end_year:04d}-{end_month:02d}", freq="M")
    for period in pd.period_range(start, end, freq="M"):
        yield period.year, period.month


def build_monthly_file_list(prefix, start_year, start_month, end_year, end_month):
    monthly_files = []
    for year, month in month_range(start_year, start_month, end_year, end_month):
        file_name = f"{prefix}{year:04d}{month:02d}.csv"
        file_path = SOURCE_DIR / file_name
        if file_path.exists():
            monthly_files.append(file_path)
    return monthly_files


def load_monthly_data(files):
    frames = []
    row_count_before_concat = 0

    for file_path in files:
        frame = pd.read_csv(file_path, low_memory=False)
        frames.append(frame)
        row_count_before_concat += len(frame)

    print(f"Row count before concatenation: {row_count_before_concat}")

    if frames:
        combined = pd.concat(frames, ignore_index=True)
    else:
        combined = pd.DataFrame()

    print(f"Row count after concatenation: {len(combined)}")
    return combined


def get_property_column(frame):
    if "PropertyType" not in frame.columns:
        raise KeyError("PropertyType column was not found in the combined dataset.")
    return "PropertyType"


def unique_property_types(frame, property_column):
    values = frame[property_column].dropna().astype(str).str.strip()
    values = values[values != ""]
    return sorted(values.unique().tolist())


def filter_residential_only(frame, property_column):
    print(f"Filtering logic applied: keep rows where {property_column} equals {PROPERTY_FILTER}.")
    before_filter = len(frame)
    filtered = frame[frame[property_column].astype(str).str.strip() == PROPERTY_FILTER].copy()
    print(f"Row count before Residential filter: {before_filter}")
    print(f"Row count after Residential filter: {len(filtered)}")
    return filtered


def build_null_summary(frame):
    summary = pd.DataFrame({
        "column": frame.columns,
        "null_count": frame.isna().sum().values,
        "row_count": len(frame),
    })
    summary["null_percent"] = np.where(
        summary["row_count"] > 0,
        (summary["null_count"] / summary["row_count"]) * 100,
        0.0,
    )
    return summary[["column", "null_count", "null_percent"]]


def build_missing_value_report(null_summary):
    flagged = null_summary[null_summary["null_percent"] > 90].copy()
    flagged.insert(0, "flag", "above_90_percent_null")
    return flagged[["flag", "column", "null_count", "null_percent"]]


def build_numeric_distribution_summary(frame, fields):
    rows = []
    for field in fields:
        if field not in frame.columns:
            rows.extend([
                {"field": field, "metric": "min", "value": np.nan},
                {"field": field, "metric": "max", "value": np.nan},
                {"field": field, "metric": "mean", "value": np.nan},
                {"field": field, "metric": "median", "value": np.nan},
                {"field": field, "metric": "p10", "value": np.nan},
                {"field": field, "metric": "p25", "value": np.nan},
                {"field": field, "metric": "p50", "value": np.nan},
                {"field": field, "metric": "p75", "value": np.nan},
                {"field": field, "metric": "p90", "value": np.nan},
            ])
            continue

        values = pd.to_numeric(frame[field], errors="coerce").dropna()
        if values.empty:
            stats = {
                "min": np.nan,
                "max": np.nan,
                "mean": np.nan,
                "median": np.nan,
                "p10": np.nan,
                "p25": np.nan,
                "p50": np.nan,
                "p75": np.nan,
                "p90": np.nan,
            }
        else:
            percentiles = np.percentile(values, PERCENTILES)
            stats = {
                "min": values.min(),
                "max": values.max(),
                "mean": values.mean(),
                "median": values.median(),
                "p10": percentiles[0],
                "p25": percentiles[1],
                "p50": percentiles[2],
                "p75": percentiles[3],
                "p90": percentiles[4],
            }

        for metric, value in stats.items():
            rows.append({"field": field, "metric": metric, "value": value})

    return pd.DataFrame(rows)


def build_report(file_name, frame, filtered_frame):
    property_column = get_property_column(frame)
    unique_types = unique_property_types(frame, property_column)
    null_summary = build_null_summary(frame)
    missing_value_report = build_missing_value_report(null_summary)
    numeric_summary = build_numeric_distribution_summary(
        frame,
        ["ClosePrice", "LivingArea", "DaysOnMarket"],
    )

    print(f"Unique property types found for {file_name}: {unique_types}")

    report_rows = []
    report_rows.append({
        "section": "metadata",
        "item": "file_name",
        "metric": "value",
        "value": file_name,
    })
    report_rows.append({
        "section": "metadata",
        "item": "source_rows",
        "metric": "count",
        "value": len(frame),
    })
    report_rows.append({
        "section": "metadata",
        "item": "filtered_rows",
        "metric": "count",
        "value": len(filtered_frame),
    })
    report_rows.append({
        "section": "filtering_logic",
        "item": property_column,
        "metric": "rule",
        "value": f'keep rows where value equals "{PROPERTY_FILTER}"',
    })
    report_rows.append({
        "section": "unique_property_types",
        "item": property_column,
        "metric": "unique_values",
        "value": ", ".join(unique_types),
    })

    for _, row in null_summary.iterrows():
        report_rows.append({
            "section": "null_summary",
            "item": row["column"],
            "metric": "null_count",
            "value": row["null_count"],
        })
        report_rows.append({
            "section": "null_summary",
            "item": row["column"],
            "metric": "null_percent",
            "value": round(float(row["null_percent"]), 2),
        })

    for _, row in missing_value_report.iterrows():
        report_rows.append({
            "section": "missing_value_report",
            "item": row["column"],
            "metric": row["flag"],
            "value": round(float(row["null_percent"]), 2),
        })

    for _, row in numeric_summary.iterrows():
        report_rows.append({
            "section": "numeric_distribution",
            "item": row["field"],
            "metric": row["metric"],
            "value": row["value"],
        })

    report_frame = pd.DataFrame(report_rows)
    print("\nNull-count summary table:")
    print(null_summary.to_string(index=False, float_format=TERMINAL_FLOAT_FORMAT))
    print("\nMissing value report (columns above 90% null):")
    print(missing_value_report.to_string(index=False, float_format=TERMINAL_FLOAT_FORMAT))
    print("\nNumeric distribution summary:")
    print(numeric_summary.to_string(index=False, float_format=TERMINAL_FLOAT_FORMAT))
    return report_frame


def save_outputs(file_path, file_name, filtered_frame, report_frame):
    file_path = Path(file_path)
    file_path.mkdir(parents=True, exist_ok=True)

    filtered_output = file_path / f"{file_name}_filtered_residential.csv"
    report_output = file_path / f"{file_name}_calculated_summary.csv"

    filtered_frame.to_csv(filtered_output, index=False)
    report_frame.to_csv(report_output, index=False)

    print(f"\nFiltered dataset saved to {filtered_output}")
    print(f"Summary report saved to {report_output}")


def process_prefix(prefix):
    end_year, end_month = most_recent_completed_month()
    monthly_files = build_monthly_file_list(prefix, START_YEAR, START_MONTH, end_year, end_month)

    if not monthly_files:
        print(f"No monthly files found for {prefix}.")
        return

    print(
        f"Found {len(monthly_files)} monthly files for {prefix} "
        f"from {START_YEAR:04d}-{START_MONTH:02d} through {end_year:04d}-{end_month:02d}."
    )

    combined = load_monthly_data(monthly_files)
    if combined.empty:
        print(f"Combined dataset for {prefix} is empty.")
        return

    filtered = filter_residential_only(combined, get_property_column(combined))
    report_frame = build_report(prefix, combined, filtered)
    save_outputs(SOURCE_DIR, prefix, filtered, report_frame)
    return filtered

# Step 4 – Merge mortgage rates onto the combined MLS datasets
def fetch_mortgage_monthly_rates(url=URL):
    mortgage = pd.read_csv(url, parse_dates=["observation_date"])
    mortgage.columns = ["date", "rate_30yr_fixed"]
    mortgage["year_month"] = mortgage["date"].dt.to_period("M")
    mortgage_monthly = (
        mortgage.groupby("year_month", as_index=False)["rate_30yr_fixed"]
        .mean()
    )
    return mortgage_monthly


def merge_mortgage_rates(frame, mortgage_monthly, date_column, dataset_name):
    frame_with_key = frame.copy()
    frame_with_key["year_month"] = pd.to_datetime(
        frame_with_key[date_column],
        errors="coerce",
    ).dt.to_period("M")

    merged = frame_with_key.merge(mortgage_monthly, on="year_month", how="left")
    null_rate_count = int(merged["rate_30yr_fixed"].isnull().sum())
    print(f"{dataset_name} null rate count after merge: {null_rate_count}")

    if null_rate_count != 0:
        raise ValueError(
            f"{dataset_name} merge produced {null_rate_count} null rate values."
        )

    return merged


def save_enriched_outputs(file_path, sold_with_rates, listings_with_rates):
    file_path = Path(file_path)
    file_path.mkdir(parents=True, exist_ok=True)

    sold_output = file_path / "CRMLSSold_filtered_residential_with_rates.csv"
    listings_output = file_path / "CRMLSListing_filtered_residential_with_rates.csv"

    sold_with_rates.to_csv(sold_output, index=False)
    listings_with_rates.to_csv(listings_output, index=False)

    print(f"Enriched sold dataset saved to {sold_output}")
    print(f"Enriched listings dataset saved to {listings_output}")


if __name__ == "__main__":
    mortgage_monthly = fetch_mortgage_monthly_rates()
    sold = process_prefix("CRMLSSold")
    listings = process_prefix("CRMLSListing")

    if sold is None or listings is None:
        raise RuntimeError("Unable to build the sold and listings datasets for merging.")

    sold_with_rates = merge_mortgage_rates(sold, mortgage_monthly, "CloseDate", "Sold")
    listings_with_rates = merge_mortgage_rates(
        listings,
        mortgage_monthly,
        "ListingContractDate",
        "Listings",
    )

    print(
        sold_with_rates[
            ["CloseDate", "year_month", "ClosePrice", "rate_30yr_fixed"]
        ].head()
    )

    save_enriched_outputs(SOURCE_DIR, sold_with_rates, listings_with_rates)

Found 24 monthly files for CRMLSSold from 2024-01 through 2026-06.
Row count before concatenation: 519932
Row count after concatenation: 519932
Filtering logic applied: keep rows where PropertyType equals Residential.
Row count before Residential filter: 519932
Row count after Residential filter: 350179
Unique property types found for CRMLSSold: ['BusinessOpportunity', 'CommercialLease', 'CommercialSale', 'Land', 'ManufacturedInPark', 'Residential', 'ResidentialIncome', 'ResidentialLease']

Null-count summary table:
                      column  null_count  null_percent
               BuyerAgentAOR       73640     14.163391
                ListAgentAOR       68018     13.082095
                    Flooring      215115     41.373680
                      ViewYN       52415     10.081126
                WaterfrontYN      519649     99.945570
                  BasementYN      511380     98.355170
               PoolPrivateYN       63195     12.154474
           OriginalListPrice        15

# WK4-5: Data Cleaning

In [ ]:
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd


SOURCE_DIR = Path("C:/Users/Viv/Documents/Career readiness/internships/IDX exchange/csv")
START_YEAR = 2024
START_MONTH = 1
PROPERTY_FILTER = "Residential"
PERCENTILES = (10, 25, 50, 75, 90)
TERMINAL_FLOAT_FORMAT = lambda value: f"{value:,.6f}"

# Step 1 – Fetch the mortgage rate data from FRED
URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(URL, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']


# Reporting rules:
# 1. Build a combined dataset from all monthly files in the date range.
# 2. Report unique PropertyType values found before filtering.
# 3. Apply the Residential-only filter.
# 4. Print and save null-count, missing-value, and numeric distribution summaries.
# 5. Save the filtered dataset as a new CSV.
#hi

def most_recent_completed_month(today=None):
    today = today or date.today()
    first_day_of_current_month = date(today.year, today.month, 1)
    last_completed_month = first_day_of_current_month - pd.offsets.MonthBegin(1)
    return last_completed_month.year, last_completed_month.month


def month_range(start_year, start_month, end_year, end_month):
    start = pd.Period(f"{start_year:04d}-{start_month:02d}", freq="M")
    end = pd.Period(f"{end_year:04d}-{end_month:02d}", freq="M")
    for period in pd.period_range(start, end, freq="M"):
        yield period.year, period.month


def build_monthly_file_list(prefix, start_year, start_month, end_year, end_month):
    monthly_files = []
    for year, month in month_range(start_year, start_month, end_year, end_month):
        file_name = f"{prefix}{year:04d}{month:02d}.csv"
        file_path = SOURCE_DIR / file_name
        if file_path.exists():
            monthly_files.append(file_path)
    return monthly_files


def load_monthly_data(files):
    frames = []
    row_count_before_concat = 0

    for file_path in files:
        frame = pd.read_csv(file_path, low_memory=False)
        frames.append(frame)
        row_count_before_concat += len(frame)

    print(f"Row count before concatenation: {row_count_before_concat}")

    if frames:
        combined = pd.concat(frames, ignore_index=True)
    else:
        combined = pd.DataFrame()

    print(f"Row count after concatenation: {len(combined)}")
    return combined


def get_property_column(frame):
    if "PropertyType" not in frame.columns:
        raise KeyError("PropertyType column was not found in the combined dataset.")
    return "PropertyType"


def dateValidate(frame,CloseDate_column, ListingContractDate_column, PurchaseDate_column): #Date consistancy check for CloseDate and ListingContractDate
    Flag = False

    
    try:
        if (frame[CloseDate_column].dropna().astype(str).str.strip() >frame[ListingContractDate_column].dropna().astype(str).str.strip() & 
            frame[PurchaseDate_column].dropna().astype(str).str.strip() < frame[CloseDate_column].dropna().astype(str).str.strip()):
            Flag = True
    except Exception as e:
        print(f"Error during date validation: {e}")
    return Flag


def GeographicValidate(frame, lat_column, long_column): #Geographic consistancy check for Latitude and Longitude

    Flag = True
    try:
        if (frame[lat_column].dropna().astype(float).between(-90, 90) & # implausible coordinates
            frame[long_column].dropna().astype(float).between(-180, 180) | 
            frame[lat_column].dropna().astype(float).isnull() & #Latitude or Longitude is null)
            frame[long_column].dropna().astype(float).isnull() |  
            frame[long_column].dropna().astype(float).greater(0) | #California coordinates should be negative
            frame[lat_column].dropna().astype(float).equals(0) & #sentinel null values
            frame[long_column].dropna().astype(float).equals(0)): 
            Flag = False
    except Exception as e:
        print(f"Error during geographic validation: {e}")
    return Flag

def unique_property_types(frame, property_column):
    values = frame[property_column].dropna().astype(str).str.strip()
    values = values[values != ""]
    return sorted(values.unique().tolist())


def convertFieldsDateTime(frame, date_columns):
    for col in date_columns:
        if col in frame.columns:
            frame[col] = pd.to_datetime(frame[col], errors="coerce")
    return frame


def filter_residential_only(frame, property_column):
    print(f"Filtering logic applied: keep rows where {property_column} equals {PROPERTY_FILTER}.")
    before_filter = len(frame)
    filtered = frame[frame[property_column].astype(str).str.strip() == PROPERTY_FILTER].copy()
    print(f"Row count before Residential filter: {before_filter}")
    print(f"Row count after Residential filter: {len(filtered)}")
    return filtered


def build_null_summary(frame):
    summary = pd.DataFrame({
        "column": frame.columns,
        "null_count": frame.isna().sum().values,
        "row_count": len(frame),
    })
    summary["null_percent"] = np.where(
        summary["row_count"] > 0,
        (summary["null_count"] / summary["row_count"]) * 100,
        0.0,
    )
    return summary[["column", "null_count", "null_percent"]]


def build_missing_value_report(null_summary):
    flagged = null_summary[null_summary["null_percent"] > 90].copy()
    flagged.insert(0, "flag", "above_90_percent_null")
    return flagged[["flag", "column", "null_count", "null_percent"]]


def build_numeric_distribution_summary(frame, fields):
    rows = []
    for field in fields:
        if field not in frame.columns:
            rows.extend([
                {"field": field, "metric": "min", "value": np.nan},
                {"field": field, "metric": "max", "value": np.nan},
                {"field": field, "metric": "mean", "value": np.nan},
                {"field": field, "metric": "median", "value": np.nan},
                {"field": field, "metric": "p10", "value": np.nan},
                {"field": field, "metric": "p25", "value": np.nan},
                {"field": field, "metric": "p50", "value": np.nan},
                {"field": field, "metric": "p75", "value": np.nan},
                {"field": field, "metric": "p90", "value": np.nan},
            ])
            continue

        values = pd.to_numeric(frame[field], errors="coerce").dropna()
        if values.empty:
            stats = {
                "min": np.nan,
                "max": np.nan,
                "mean": np.nan,
                "median": np.nan,
                "p10": np.nan,
                "p25": np.nan,
                "p50": np.nan,
                "p75": np.nan,
                "p90": np.nan,
            }
        else:
            percentiles = np.percentile(values, PERCENTILES)
            stats = {
                "min": values.min(),
                "max": values.max(),
                "mean": values.mean(),
                "median": values.median(),
                "p10": percentiles[0],
                "p25": percentiles[1],
                "p50": percentiles[2],
                "p75": percentiles[3],
                "p90": percentiles[4],
            }

        for metric, value in stats.items():
            rows.append({"field": field, "metric": metric, "value": value})

    return pd.DataFrame(rows)


def build_report(file_name, frame, filtered_frame):
    property_column = get_property_column(frame)
    unique_types = unique_property_types(frame, property_column)
    null_summary = build_null_summary(frame)
    missing_value_report = build_missing_value_report(null_summary)
    numeric_summary = build_numeric_distribution_summary(
        frame,
        ["ClosePrice", "LivingArea", "DaysOnMarket"],
    )

    print(f"Unique property types found for {file_name}: {unique_types}")

    report_rows = []
    report_rows.append({
        "section": "metadata",
        "item": "file_name",
        "metric": "value",
        "value": file_name,
    })
    report_rows.append({
        "section": "metadata",
        "item": "source_rows",
        "metric": "count",
        "value": len(frame),
    })
    report_rows.append({
        "section": "metadata",
        "item": "filtered_rows",
        "metric": "count",
        "value": len(filtered_frame),
    })
    report_rows.append({
        "section": "filtering_logic",
        "item": property_column,
        "metric": "rule",
        "value": f'keep rows where value equals "{PROPERTY_FILTER}"',
    })
    report_rows.append({
        "section": "unique_property_types",
        "item": property_column,
        "metric": "unique_values",
        "value": ", ".join(unique_types),
    })

    for _, row in null_summary.iterrows():
        report_rows.append({
            "section": "null_summary",
            "item": row["column"],
            "metric": "null_count",
            "value": row["null_count"],
        })
        report_rows.append({
            "section": "null_summary",
            "item": row["column"],
            "metric": "null_percent",
            "value": round(float(row["null_percent"]), 2),
        })

    for _, row in missing_value_report.iterrows():
        report_rows.append({
            "section": "missing_value_report",
            "item": row["column"],
            "metric": row["flag"],
            "value": round(float(row["null_percent"]), 2),
        })

    for _, row in numeric_summary.iterrows():
        report_rows.append({
            "section": "numeric_distribution",
            "item": row["field"],
            "metric": row["metric"],
            "value": row["value"],
        })

    report_frame = pd.DataFrame(report_rows)
    print("\nNull-count summary table:")
    print(null_summary.to_string(index=False, float_format=TERMINAL_FLOAT_FORMAT))
    print("\nMissing value report (columns above 90% null):")
    print(missing_value_report.to_string(index=False, float_format=TERMINAL_FLOAT_FORMAT))
    print("\nNumeric distribution summary:")
    print(numeric_summary.to_string(index=False, float_format=TERMINAL_FLOAT_FORMAT))
    return report_frame


def save_outputs(file_path, file_name, filtered_frame, report_frame):
    file_path = Path(file_path)
    file_path.mkdir(parents=True, exist_ok=True)

    filtered_output = file_path / f"{file_name}_filtered_residential.csv"
    report_output = file_path / f"{file_name}_calculated_summary.csv"

    filtered_frame.to_csv(filtered_output, index=False)
    report_frame.to_csv(report_output, index=False)

    print(f"\nFiltered dataset saved to {filtered_output}")
    print(f"Summary report saved to {report_output}")


def process_prefix(prefix):
    end_year, end_month = most_recent_completed_month()
    monthly_files = build_monthly_file_list(prefix, START_YEAR, START_MONTH, end_year, end_month)

    if not monthly_files:
        print(f"No monthly files found for {prefix}.")
        return

    print(
        f"Found {len(monthly_files)} monthly files for {prefix} "
        f"from {START_YEAR:04d}-{START_MONTH:02d} through {end_year:04d}-{end_month:02d}."
    )

    combined = load_monthly_data(monthly_files)
    if combined.empty:
        print(f"Combined dataset for {prefix} is empty.")
        return

    filtered = filter_residential_only(combined, get_property_column(combined))
    report_frame = build_report(prefix, combined, filtered)
    save_outputs(SOURCE_DIR, prefix, filtered, report_frame)
    return filtered

# Step 4 – Merge mortgage rates onto the combined MLS datasets
def fetch_mortgage_monthly_rates(url=URL):
    mortgage = pd.read_csv(url, parse_dates=["observation_date"])
    mortgage.columns = ["date", "rate_30yr_fixed"]
    mortgage["year_month"] = mortgage["date"].dt.to_period("M")
    mortgage_monthly = (
        mortgage.groupby("year_month", as_index=False)["rate_30yr_fixed"]
        .mean()
    )
    return mortgage_monthly


def merge_mortgage_rates(frame, mortgage_monthly, date_column, dataset_name):
    frame_with_key = frame.copy()
    frame_with_key["year_month"] = pd.to_datetime(
        frame_with_key[date_column],
        errors="coerce",
    ).dt.to_period("M")

    merged = frame_with_key.merge(mortgage_monthly, on="year_month", how="left")
    null_rate_count = int(merged["rate_30yr_fixed"].isnull().sum())
    print(f"{dataset_name} null rate count after merge: {null_rate_count}")

    if null_rate_count != 0:
        raise ValueError(
            f"{dataset_name} merge produced {null_rate_count} null rate values."
        )

    return merged


def save_enriched_outputs(file_path, sold_with_rates, listings_with_rates):
    file_path = Path(file_path)
    file_path.mkdir(parents=True, exist_ok=True)

    sold_output = file_path / "CRMLSSold_filtered_residential_with_rates.csv"
    listings_output = file_path / "CRMLSListing_filtered_residential_with_rates.csv"

    sold_with_rates.to_csv(sold_output, index=False)
    listings_with_rates.to_csv(listings_output, index=False)

    print(f"Enriched sold dataset saved to {sold_output}")
    print(f"Enriched listings dataset saved to {listings_output}")


if __name__ == "__main__":
    mortgage_monthly = fetch_mortgage_monthly_rates()
    sold = process_prefix("CRMLSSold")
    listings = process_prefix("CRMLSListing")

    if sold is None or listings is None:
        raise RuntimeError("Unable to build the sold and listings datasets for merging.")

    sold_with_rates = merge_mortgage_rates(sold, mortgage_monthly, "CloseDate", "Sold")
    listings_with_rates = merge_mortgage_rates(
        listings,
        mortgage_monthly,
        "ListingContractDate",
        "Listings",
    )

    print(
        sold_with_rates[
            ["CloseDate", "year_month", "ClosePrice", "rate_30yr_fixed"]
        ].head()
    )

    save_enriched_outputs(SOURCE_DIR, sold_with_rates, listings_with_rates)